# SWEAチャットボット v2
### スライド窓アンサンブル法（N乗融合）による長文対応チャットボット
- 会話が短い間は通常生成（速い）
- 会話が長くなったら自動でN乗融合アンサンブルに切り替え（記憶が消えない）
- GradioのスライダーでENSEMBLE_Nをリアルタイム調整可能

In [ ]:
!pip install transformers torch accelerate gradio -q

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("モデルを読み込んでいます...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("完了")

In [ ]:
# ===== 設定 =====
SINK_SIZE      = 10
WINDOW_SIZE    = 512
LOCAL_SIZE     = 100
BATCH_SIZE     = 16
MIN_TOKENS     = SINK_SIZE + WINDOW_SIZE + LOCAL_SIZE  # 622
MAX_NEW_TOKENS = 200
ENSEMBLE_N     = 2.0  # N乗融合の乗数（Gradioスライダーで上書き可）

SYSTEM_PROMPT = "あなたは親切なアシスタントです。日本語で簡潔に答えてください。"

print(f"設定完了")
print(f"アンサンブル切り替え閾値: {MIN_TOKENS} トークン")
print(f"N乗融合の乗数: {ENSEMBLE_N}")


In [ ]:
import torch.nn.functional as F

def build_windows(total_len):
    stride  = WINDOW_SIZE // 2
    mid_end = total_len - LOCAL_SIZE
    windows = []
    pos = SINK_SIZE
    while pos + WINDOW_SIZE <= mid_end:
        windows.append((pos, pos + WINDOW_SIZE))
        pos += stride
    last_start = mid_end - WINDOW_SIZE
    last_end   = mid_end
    if last_start >= SINK_SIZE:
        if len(windows) == 0 or windows[-1] != (last_start, last_end):
            windows.append((last_start, last_end))
    return windows


def predict_next_logits(input_ids, n=None):
    """
    次のトークンの予測を返す
    n=None のときはグローバル ENSEMBLE_N を使用
    """
    _n = n if n is not None else ENSEMBLE_N
    total_len = input_ids.shape[1]
    mid_end   = total_len - LOCAL_SIZE

    # 通常生成
    if total_len < MIN_TOKENS:
        with torch.no_grad():
            out = model(input_ids)
        return out.logits[:, -1, :], "通常生成"

    # 全窓のinput_idsを作る
    windows = build_windows(total_len)
    all_window_ids = [
        torch.cat([
            input_ids[:, :SINK_SIZE],
            input_ids[:, w_start:w_end],
            input_ids[:, mid_end:]
        ], dim=1)
        for w_start, w_end in windows
    ]

    # ミニバッチに分けて処理し、確率のN乗を集める
    all_probs = []
    for i in range(0, len(all_window_ids), BATCH_SIZE):
        mini_batch = torch.cat(all_window_ids[i:i+BATCH_SIZE], dim=0)
        with torch.no_grad():
            out = model(mini_batch)
        probs = F.softmax(out.logits[:, -1, :], dim=-1)
        all_probs.append(probs ** _n)  # N乗融合

    # 全窓の確率N乗を平均して正規化
    fused = torch.cat(all_probs, dim=0).mean(dim=0, keepdim=True)
    fused = fused / fused.sum(dim=-1, keepdim=True)
    return fused, f"アンサンブル（窓数{len(windows)}, N={_n}）"


def build_prompt(history, user_message):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for human, assistant in history:
        messages.append({"role": "user",      "content": human})
        messages.append({"role": "assistant", "content": assistant})
    messages.append({"role": "user", "content": user_message})
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return prompt


def generate_response(history, user_message, ensemble_n=None):
    prompt    = build_prompt(history, user_message)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    generated = input_ids.clone()
    token_count = input_ids.shape[1]
    mode_log = ""

    for _ in range(MAX_NEW_TOKENS):
        out, mode = predict_next_logits(generated, n=ensemble_n)
        if not mode_log:
            mode_log = mode
        next_token_id = out.argmax(dim=-1, keepdim=True)
        if next_token_id.item() == tokenizer.eos_token_id:
            break
        generated = torch.cat([generated, next_token_id], dim=1)

    response = tokenizer.decode(
        generated[0, input_ids.shape[1]:],
        skip_special_tokens=True
    )
    debug_info = f"\n\n---\nトークン数: {token_count} | モード: {mode_log}"
    return response + debug_info


print("関数の定義完了")


In [ ]:
import gradio as gr

def chat(message, history, ensemble_n):
    response = generate_response(history, message, ensemble_n=ensemble_n)
    return response


with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# SWEAチャットボット v2")
    gr.Markdown(
        "スライド窓アンサンブル法（N乗融合）による長文対応チャットボット  \n"
        "会話が622トークンを超えると自動でアンサンブルモードに切り替わります"
    )

    with gr.Row():
        n_slider = gr.Slider(
            minimum=1.0, maximum=10.0, value=2.0, step=0.5,
            label="ENSEMBLE_N（1.0=通常平均　大きいほど確信度重視）"
        )

    gr.ChatInterface(
        fn=chat,
        additional_inputs=[n_slider],
        examples=[
            ["こんにちは！あなたは何ができますか？"],
            ["Pythonでフィボナッチ数列を計算するコードを書いてください"],
            ["東京のおすすめ観光地を教えてください"],
        ],
    )

demo.launch(share=True)
